# Phase 3 — Integration E2E Test (TASK-14)

Notebook này chạy 15 câu hỏi thử nghiệm qua toàn bộ pipeline RAG:

```
question → QueryPlanner → SubgraphExtractor → HybridSearch → ContextAssembler → AnswerGenerator
```

**Phạm vi [A]:** Đất đai (chuyển mục đích SDĐ + cấp sổ đỏ lần đầu), TP.HCM + Đồng Nai + Toàn quốc  
**DoD cần đạt:**
- DoD 1: Câu hỏi Đất đai TP.HCM trả lời trong < 30s
- DoD 2: ≥ 2 câu hỏi cho mỗi thủ tục
- DoD 3: Câu hỏi thiếu jurisdiction → `confirmation_needed=True`
- DoD 4: Negative test khai sinh TP.HCM vs Đồng Nai → không bịa sự khác biệt
- DoD 5: Ghi kết quả + nhận xét vào notebook

> **Lưu ý citation**: LLM có thể dùng format tắt `[Điều X, Luật Y]` thay vì format chuẩn `[Điều X, Văn bản Y]`.  
> `parse_citations()` chỉ bắt format chuẩn — citation count = 0 không có nghĩa LLM không trích dẫn.  
> Đánh giá chất lượng trích dẫn bằng mắt qua phần TRẢ LỜI.

In [1]:
import os
import sys
import json
import time
import logging
from pathlib import Path

# Tìm project root bất kể notebook được chạy từ đâu
_cwd = Path.cwd()
_project_root = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / 'CLAUDE.md').exists()),
    _cwd,
)
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))
print(f'Project root: {_project_root}')

from dotenv import load_dotenv
load_dotenv(_project_root / '.env')

logging.basicConfig(level=logging.WARNING)
print('Import OK')

Project root: .../vn-legal-graphrag
Import OK


In [2]:
import anthropic
from neo4j import GraphDatabase
from qdrant_client import QdrantClient

from src.ingestion.vectorizer import load_model
from src.pipeline import run_pipeline

# Khởi tạo clients một lần, dùng lại cho tất cả câu hỏi
neo4j_driver = GraphDatabase.driver(
    os.getenv('NEO4J_URI', 'bolt://localhost:7687'),
    auth=(os.getenv('NEO4J_USER', 'neo4j'), os.getenv('NEO4J_PASSWORD', '')),
)
qdrant_client = QdrantClient(
    host=os.getenv('QDRANT_HOST', 'localhost'),
    port=int(os.getenv('QDRANT_PORT', '6333')),
)
anthropic_client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
model = load_model()

print('Clients OK')

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Clients OK


In [3]:
results = []  # lưu kết quả để đánh giá cuối

def ask(question: str, label: str = '') -> dict:
    """Chạy pipeline và in kết quả gọn."""
    print(f"\n{'='*70}")
    if label:
        print(f"[{label}]")
    print(f"CÂU HỎI: {question}")
    print('='*70)

    result = run_pipeline(
        question,
        neo4j_driver=neo4j_driver,
        qdrant_client=qdrant_client,
        anthropic_client=anthropic_client,
        model=model,
    )

    if result['confirmation_needed']:
        print('⚠️  CẦN XÁC NHẬN:')
        print(result['confirmation_prompt'])
    else:
        print(f"📊 LCCIDs: {result['lccids_count']}  |  Top-k: {result['top_k_count']}  |  Context: ~{result['context_tokens']} tokens")
        print(f"\n💬 TRẢ LỜI:\n{result['answer']}")
        if result['citations']:
            print(f"\n📌 CITATIONS parsed ({len(result['citations'])}): {result['citations']}")
        else:
            print('\n📌 CITATIONS parsed: 0 (LLM có thể dùng format tắt — kiểm tra thủ công)')

    print(f"\n⏱️  {result['elapsed_seconds']}s")
    return result

print('ask() ready')

ask() ready


## 1. Chuyển mục đích sử dụng đất — TP.HCM (DoD 1 + DoD 2)

In [4]:
# DoD 1: câu hỏi chuẩn, phải trả lời trong < 30s
r = ask(
    'Điều kiện để chuyển mục đích sử dụng đất tại TP.HCM là gì?',
    label='Q01 | CMĐSDĐ TP.HCM | DoD-1'
)
results.append(r)

# DoD 1 checks
assert r['elapsed_seconds'] < 45, f'TIMEOUT: {r["elapsed_seconds"]}s'
assert not r['confirmation_needed'], 'Câu hỏi có đủ jurisdiction — không được confirmation_needed'
# Citation count là soft check vì LLM có thể dùng format tắt
if len(r['citations']) >= 1:
    print('✅ DoD 1 PASS (có citation theo format chuẩn)')
else:
    print('⚠️  DoD 1 PARTIAL — dưới 30s, có trả lời, nhưng citation format chưa chuẩn (kiểm tra thủ công)')


[Q01 | CMĐSDĐ TP.HCM | DoD-1]
CÂU HỎI: Điều kiện để chuyển mục đích sử dụng đất tại TP.HCM là gì?


📊 LCCIDs: 13  |  Top-k: 22  |  Context: ~5667 tokens

💬 TRẢ LỜI:
# Điều kiện chuyển mục đích sử dụng đất tại TP.HCM

Dựa trên các quy định hiện hành, điều kiện chuyển mục đích sử dụng đất bao gồm các nhóm sau:

---

## 1. Quy trình và thủ tục chung

- **Người sử dụng đất phải nộp hồ sơ** đề nghị chuyển mục đích sử dụng đất theo quy định. [Điều 227, Khoản 1, Văn bản luat-dat-dai-2024]
- **Cơ quan quản lý đất đai kiểm tra** các điều kiện chuyển mục đích; nếu hồ sơ chưa đủ thì hướng dẫn bổ sung. [Điều 227, Khoản 2, Văn bản luat-dat-dai-2024]
- Trường hợp **không thuộc diện phải xin phép** (theo khoản 1 Điều 121 Luật Đất đai) thì không cần xin phép cơ quan nhà nước có thẩm quyền. [Điều 121, Khoản 3, Văn bản luat-dat-dai-2024]

---

## 2. Điều kiện đặc thù khi chuyển từ đất rừng

Khi chuyển đất rừng phòng hộ, rừng đặc dụng, rừng sản xuất sang mục đích khác, phải:
- Có **phương án trồng rừng thay thế** hoặc **văn bản hoàn thành trách nhiệm nộp tiền trồng rừng thay thế** theo pháp luật lâm ng

In [5]:
r = ask(
    'Hộ gia đình có được chuyển đất nông nghiệp sang đất ở tại TP.HCM không? Cần điều kiện gì?',
    label='Q02 | CMĐSDĐ TP.HCM — điều kiện hộ gia đình'
)
results.append(r)


[Q02 | CMĐSDĐ TP.HCM — điều kiện hộ gia đình]
CÂU HỎI: Hộ gia đình có được chuyển đất nông nghiệp sang đất ở tại TP.HCM không? Cần điều kiện gì?


📊 LCCIDs: 13  |  Top-k: 22  |  Context: ~5912 tokens

💬 TRẢ LỜI:
# Chuyển đất nông nghiệp sang đất ở tại TP.HCM

## 1. Cơ sở pháp lý và điều kiện chung

Hộ gia đình, cá nhân **được phép** chuyển mục đích sử dụng đất từ đất nông nghiệp sang đất ở, tuy nhiên phải đáp ứng các điều kiện theo quy định.

Theo quy định hiện hành, việc chuyển mục đích áp dụng cho các trường hợp:
- Đất vườn, ao, đất nông nghiệp **trong cùng thửa đất có đất ở** được xác định khi công nhận quyền sử dụng đất; hoặc
- Đất có **nguồn gốc là đất vườn, ao gắn liền đất ở** nhưng người sử dụng đất tách ra để chuyển quyền sử dụng đất; hoặc
- Đất do đơn vị đo đạc khi đo vẽ bản đồ địa chính **trước ngày 01/7/2014** đã tự đo đạc tách thành các thửa riêng.

[Điều 10, Khoản 2, Điểm c, Văn bản nghi-quyet-254-2025-qh15]

---

## 2. Nghĩa vụ tài chính (tiền sử dụng đất)

Khi chuyển mục đích, hộ gia đình phải nộp tiền sử dụng đất theo **3 mức** tính trên chênh lệch giữa giá đất ở và giá đất nông nghiệp tại thời điểm có quyết định 

In [6]:
r = ask(
    'Hồ sơ xin chuyển mục đích sử dụng đất tại TP.HCM gồm những giấy tờ gì?',
    label='Q03 | CMĐSDĐ TP.HCM — hồ sơ'
)
results.append(r)


[Q03 | CMĐSDĐ TP.HCM — hồ sơ]
CÂU HỎI: Hồ sơ xin chuyển mục đích sử dụng đất tại TP.HCM gồm những giấy tờ gì?


📊 LCCIDs: 13  |  Top-k: 22  |  Context: ~5647 tokens

💬 TRẢ LỜI:
## Hồ sơ xin chuyển mục đích sử dụng đất tại TP.HCM

Dựa trên thông tin trong CONTEXT, có thể cung cấp một số quy định liên quan, tuy nhiên **CONTEXT không liệt kê đầy đủ thành phần hồ sơ cụ thể** cho thủ tục chuyển mục đích sử dụng đất. Dưới đây là những gì có thể xác định được:

---

### 1. Trình tự chung theo Luật Đất đai 2024

- **Bước 1:** Người sử dụng đất nộp hồ sơ đề nghị chuyển mục đích sử dụng đất theo quy định. [Điều 227, Khoản 1, Văn bản luat-dat-dai-2024]
- **Bước 2:** Cơ quan có chức năng quản lý đất đai kiểm tra các điều kiện chuyển mục đích sử dụng đất. Nếu hồ sơ chưa đảm bảo, cơ quan sẽ hướng dẫn bổ sung. [Điều 227, Khoản 2, Văn bản luat-dat-dai-2024]

---

### 2. Trường hợp chuyển từ đất chuyên trồng lúa sang phi nông nghiệp

Nếu thửa đất là **đất chuyên trồng lúa**, hồ sơ phải bổ sung thêm **phương án sử dụng tầng đất mặt**, bao gồm:
- Thông tin của người được giao đất/cho thuê đất/cho phép chuyển mục đ

In [7]:
r = ask(
    'Nghĩa vụ tài chính khi chuyển mục đích sử dụng đất sang đất ở tại TP.HCM là bao nhiêu?',
    label='Q04 | CMĐSDĐ TP.HCM — nghĩa vụ tài chính'
)
results.append(r)


[Q04 | CMĐSDĐ TP.HCM — nghĩa vụ tài chính]
CÂU HỎI: Nghĩa vụ tài chính khi chuyển mục đích sử dụng đất sang đất ở tại TP.HCM là bao nhiêu?


📊 LCCIDs: 13  |  Top-k: 22  |  Context: ~5617 tokens

💬 TRẢ LỜI:
## Nghĩa vụ tài chính khi chuyển mục đích sử dụng đất sang đất ở tại TP.HCM

### 1. Mức thu tiền sử dụng đất khi chuyển mục đích

Đối với trường hợp **đất vườn, ao, đất nông nghiệp trong cùng thửa đất có đất ở** (hoặc đất có nguồn gốc vườn ao gắn liền đất ở bị tách ra), mức thu tiền sử dụng đất được tính theo **chênh lệch** giữa tiền sử dụng đất tính theo giá đất ở và tiền sử dụng đất tính theo giá đất nông nghiệp, cụ thể:

| Phần diện tích | Mức thu |
|---|---|
| Trong hạn mức giao đất ở tại địa phương | **30%** chênh lệch |
| Vượt hạn mức nhưng không quá 1 lần hạn mức | **50%** chênh lệch |
| Vượt quá 1 lần hạn mức | **100%** chênh lệch |

> ⚠️ Mức thu ưu đãi này **chỉ được tính một lần cho một hộ gia đình, cá nhân** (tính trên 01 thửa đất).

[Điều 10, Khoản 2, Điểm c, Văn bản nghi-quyet-254-2025-qh15]

---

### 2. Xác định hạn mức giao đất ở để tính tiền sử dụng đất

Hạn mức giao đất ở được xác định theo quy định của c

## 2. Chuyển mục đích sử dụng đất — Đồng Nai

In [8]:
r = ask(
    'Quy trình chuyển mục đích sử dụng đất nông nghiệp sang đất ở tại Đồng Nai như thế nào?',
    label='Q05 | CMĐSDĐ Đồng Nai — quy trình'
)
results.append(r)


[Q05 | CMĐSDĐ Đồng Nai — quy trình]
CÂU HỎI: Quy trình chuyển mục đích sử dụng đất nông nghiệp sang đất ở tại Đồng Nai như thế nào?


📊 LCCIDs: 13  |  Top-k: 22  |  Context: ~5056 tokens

💬 TRẢ LỜI:
# Quy trình chuyển mục đích sử dụng đất nông nghiệp sang đất ở tại Đồng Nai

## I. Điều kiện cơ bản

Việc chuyển mục đích sử dụng đất nông nghiệp sang đất ở phải **phù hợp với quy hoạch sử dụng đất cấp huyện** hoặc quy hoạch chung/phân khu/xây dựng/nông thôn. [Điều 141, Khoản 4, Điểm c, Văn bản luat-dat-dai-2024]

---

## II. Nghĩa vụ tài chính khi chuyển mục đích

### 1. Trường hợp đất vườn, ao, đất nông nghiệp trong cùng thửa có đất ở (hoặc đất vườn ao gắn liền đất ở)

Tiền sử dụng đất được tính theo **chênh lệch** giữa giá đất ở và giá đất nông nghiệp tại thời điểm có quyết định cho phép chuyển mục đích, cụ thể: [Điều 10, Khoản 2, Điểm c, Văn bản nghi-quyet-254-2025-qh15]

| Diện tích chuyển mục đích | Mức thu |
|---|---|
| Trong hạn mức giao đất ở tại địa phương | **30%** chênh lệch |
| Vượt hạn mức nhưng không quá 1 lần hạn mức | **50%** chênh lệch |
| Vượt quá 1 lần hạn mức | **100%** chênh lệch |

> ⚠️ **Lưu ý quan

In [9]:
r = ask(
    'Thời hạn giải quyết hồ sơ chuyển mục đích sử dụng đất tại Đồng Nai là bao lâu?',
    label='Q06 | CMĐSDĐ Đồng Nai — thời hạn'
)
results.append(r)


[Q06 | CMĐSDĐ Đồng Nai — thời hạn]
CÂU HỎI: Thời hạn giải quyết hồ sơ chuyển mục đích sử dụng đất tại Đồng Nai là bao lâu?


📊 LCCIDs: 13  |  Top-k: 22  |  Context: ~5058 tokens

💬 TRẢ LỜI:
## Thời hạn giải quyết hồ sơ chuyển mục đích sử dụng đất tại Đồng Nai

Dựa trên các tài liệu trong hệ thống, **context hiện tại không có đủ thông tin** để trả lời cụ thể về thời hạn giải quyết hồ sơ chuyển mục đích sử dụng đất tại tỉnh Đồng Nai.

### Những gì context có thể cung cấp liên quan:

Theo quy định hiện hành, **Ủy ban nhân dân cấp tỉnh** (bao gồm UBND tỉnh Đồng Nai) có trách nhiệm ban hành quy định về trình tự, thủ tục hành chính về đất đai — trong đó bao gồm **thời gian tối đa thực hiện từng bước** — **chậm nhất đến ngày 01 tháng 7 năm 2026**. [Điều 15, Khoản 1, Văn bản nghi-dinh-49-2026-nd-cp]

Trong thời gian chưa ban hành quy định mới, UBND cấp tỉnh quyết định việc áp dụng trình tự, thủ tục theo quy định pháp luật trước ngày Nghị định số 49/2026/NĐ-CP có hiệu lực, hoặc quyết định theo từng trường hợp cụ thể. [Điều 15, Khoản 1, Văn bản nghi-dinh-49-2026-nd-cp]

### Khuyến nghị:

Để biết thời hạn giải quyết hồ

## 3. Cấp sổ đỏ lần đầu — TP.HCM (DoD 2)

In [10]:
r = ask(
    'Điều kiện để được cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại TP.HCM là gì?',
    label='Q07 | Cấp sổ đỏ TP.HCM — điều kiện'
)
results.append(r)


[Q07 | Cấp sổ đỏ TP.HCM — điều kiện]
CÂU HỎI: Điều kiện để được cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại TP.HCM là gì?


📊 LCCIDs: 13  |  Top-k: 22  |  Context: ~5875 tokens

💬 TRẢ LỜI:
# Điều kiện cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại TP.HCM

## 1. Nguyên tắc chung

Giấy chứng nhận chỉ được cấp sau khi người sử dụng đất **hoàn thành nghĩa vụ tài chính** theo quy định, trừ trường hợp không thuộc đối tượng phải thực hiện, được miễn hoặc được ghi nợ nghĩa vụ tài chính.
[Điều 135, Khoản 3, Văn bản luat-dat-dai-2024]

---

## 2. Điều kiện theo từng trường hợp

### A. Trường hợp có giấy tờ về quyền sử dụng đất

Hộ gia đình, cá nhân có các giấy tờ hợp lệ được cấp Giấy chứng nhận, bao gồm:
- Giấy tờ tạm giao đất của UBND cấp huyện, cấp tỉnh
- Đơn đề nghị sử dụng đất được UBND cấp xã, hợp tác xã nông nghiệp phê duyệt trước ngày 01/7/1980, hoặc được UBND cấp huyện, cấp tỉnh phê duyệt

[Điều 137, Khoản 1, Điểm l, Văn bản luat-dat-dai-2024]

### B. Trường hợp thửa đất có diện tích nhỏ hơn diện tích tối thiểu

Thửa đất được hình thành **trước ngày** văn bản quy định của UBND cấp tỉnh về diện tích tối thi

In [11]:
r = ask(
    'Hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại TP.HCM gồm những gì?',
    label='Q08 | Cấp sổ đỏ TP.HCM — hồ sơ'
)
results.append(r)


[Q08 | Cấp sổ đỏ TP.HCM — hồ sơ]
CÂU HỎI: Hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại TP.HCM gồm những gì?


📊 LCCIDs: 13  |  Top-k: 22  |  Context: ~5905 tokens

💬 TRẢ LỜI:
## Hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại TP.HCM

Dựa trên thông tin trong CONTEXT được cung cấp, tôi chỉ tìm thấy một số quy định liên quan đến **giấy tờ chứng minh quyền sử dụng đất** và **trình tự thủ tục** ở mức khái quát. Cụ thể:

---

### 1. Giấy tờ về quyền sử dụng đất (làm căn cứ cấp GCN)

Người sử dụng đất cần có **một trong các loại giấy tờ** sau:

**Giấy tờ từ chế độ cũ:**
- Bằng khoán điền thổ; Văn tự đoạn mãi bất động sản có chứng nhận; Văn tự mua bán/tặng cho/đổi/thừa kế nhà ở gắn liền với đất ở có chứng nhận; Bản di chúc hoặc giấy thỏa thuận tương phân di sản về nhà ở; Giấy phép xây cất nhà ở; Giấy phép hợp thức hóa kiến trúc; Bản án Tòa án thuộc chế độ cũ đã có hiệu lực.
[Điều 137, Khoản 1, Điểm b, Văn bản luat-dat-dai-2024]

**Giấy tờ từ quá trình đăng ký ruộng đất (Chỉ thị 299/TTg năm 1980):**
- Biên bản xét duyệt của Hội đồng đăng ký ruộng đất cấp xã; Bản tổng hợp các trường hợp

In [12]:
r = ask(
    'Trường hợp đất không có giấy tờ tại TP.HCM thì có được cấp sổ đỏ lần đầu không? Điều kiện là gì?',
    label='Q09 | Cấp sổ đỏ TP.HCM — đất không giấy tờ'
)
results.append(r)


[Q09 | Cấp sổ đỏ TP.HCM — đất không giấy tờ]
CÂU HỎI: Trường hợp đất không có giấy tờ tại TP.HCM thì có được cấp sổ đỏ lần đầu không? Điều kiện là gì?


📊 LCCIDs: 13  |  Top-k: 22  |  Context: ~5721 tokens

💬 TRẢ LỜI:
# Cấp sổ đỏ lần đầu cho đất không có giấy tờ tại TP.HCM

## 1. Có được cấp sổ đỏ không?

**Có thể được cấp**, nếu đáp ứng các điều kiện theo quy định. Pháp luật hiện hành có quy định cụ thể cho trường hợp hộ gia đình, cá nhân sử dụng đất không có giấy tờ mà không vi phạm pháp luật đất đai.

---

## 2. Điều kiện cụ thể

### Trường hợp sử dụng đất **trước ngày 18/12/1980**

Được cấp Giấy chứng nhận nếu:
- Được **UBND cấp xã nơi có đất xác nhận không có tranh chấp**

[Điều 138, Khoản 1, Văn bản luat-dat-dai-2024]

---

### Xử lý các phần diện tích đất

**a) Phần diện tích đất còn lại** (sau khi xác định diện tích đất ở theo quy định):
- Nếu hiện trạng là **đất nông nghiệp**: được công nhận theo hình thức **Nhà nước giao đất không thu tiền sử dụng đất**
- Nếu người sử dụng đất có nhu cầu chuyển sang **đất phi nông nghiệp** và phù hợp với quy hoạch sử dụng đất cấp huyện, quy hoạch sử dụng đất cấp xã hoặc một trong các quy hoạc

## 4. Cấp sổ đỏ lần đầu — Đồng Nai

In [13]:
r = ask(
    'Cơ quan nào tiếp nhận hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại Đồng Nai?',
    label='Q10 | Cấp sổ đỏ Đồng Nai — cơ quan'
)
results.append(r)


[Q10 | Cấp sổ đỏ Đồng Nai — cơ quan]
CÂU HỎI: Cơ quan nào tiếp nhận hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại Đồng Nai?


📊 LCCIDs: 13  |  Top-k: 22  |  Context: ~5889 tokens

💬 TRẢ LỜI:
## Cơ quan tiếp nhận hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại Đồng Nai

Dựa trên các tài liệu trong CONTEXT, **không có quy định cụ thể** nào xác định rõ tên cơ quan tiếp nhận hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại tỉnh Đồng Nai.

Tuy nhiên, CONTEXT có đề cập một số thông tin liên quan:

- **Văn phòng đăng ký đất đai / Chi nhánh Văn phòng đăng ký đất đai** được nhắc đến với tư cách là cơ quan có trách nhiệm trong quá trình giải quyết thủ tục đăng ký đất đai, tài sản gắn liền với đất [Điều 19, Khoản 4, Văn bản nghi-dinh-101-2024-nd-cp].

- Nghị quyết số 22/2024/NQ-HĐND của tỉnh Đồng Nai quy định về **phí thẩm định hồ sơ cấp Giấy chứng nhận** (cả hình thức nộp **trực tiếp** và **trực tuyến**), nhưng không nêu rõ tên cơ quan tiếp nhận cụ thể [Phụ lục I, Văn bản nghi-quyet-22-2024-nq-hdnd-dong-nai].

---

**Kết luận:** Context hiện tại **không đủ thông tin** để trả lời đầy đủ câ

In [14]:
r = ask(
    'Phí và lệ phí khi đăng ký cấp sổ đỏ lần đầu tại Đồng Nai là bao nhiêu?',
    label='Q11 | Cấp sổ đỏ Đồng Nai — phí lệ phí'
)
results.append(r)


[Q11 | Cấp sổ đỏ Đồng Nai — phí lệ phí]
CÂU HỎI: Phí và lệ phí khi đăng ký cấp sổ đỏ lần đầu tại Đồng Nai là bao nhiêu?


📊 LCCIDs: 13  |  Top-k: 22  |  Context: ~5760 tokens

💬 TRẢ LỜI:
## Phí và lệ phí khi đăng ký cấp sổ đỏ lần đầu tại Đồng Nai

### 1. Phí thẩm định hồ sơ (cấp lần đầu)

Theo [Phụ lục I, Văn bản nghi-quyet-22-2024-nq-hdnd-dong-nai], mức phí thẩm định hồ sơ cấp Giấy chứng nhận lần đầu như sau:

| Loại hồ sơ | Hộ gia đình/cá nhân - Trực tiếp | Hộ gia đình/cá nhân - Trực tuyến | Tổ chức - Trực tiếp | Tổ chức - Trực tuyến |
|---|:---:|:---:|:---:|:---:|
| Chỉ quyền sử dụng đất | 880.000 đ | 836.000 đ | 1.260.000 đ | 1.197.000 đ |
| Chỉ tài sản gắn liền với đất | 980.000 đ | 931.000 đ | 1.840.000 đ | 1.748.000 đ |
| Cả đất và tài sản gắn liền với đất | 1.250.000 đ | 1.187.500 đ | 2.090.000 đ | 1.985.500 đ |

**Lưu ý đặc biệt:**
- Một thửa đất có **nhiều người đồng sử dụng**: mỗi Giấy chứng nhận cấp thêm thu **50.000 đồng/GCN/người** (áp dụng cả trực tiếp lẫn trực tuyến). [Phụ lục I, Văn bản nghi-quyet-22-2024-nq-hdnd-dong-nai]
- Tổ chức có dự án **nhiều thửa đất**: từ thửa thứ hai trở đi thu 

## 5. DoD 3 — Thiếu jurisdiction → confirmation_needed

In [15]:
r = ask(
    'Điều kiện chuyển mục đích sử dụng đất nông nghiệp sang đất ở là gì?',
    label='Q12 | Thiếu jurisdiction | DoD-3'
)
results.append(r)
assert r['confirmation_needed'] is True, 'Phải confirmation_needed=True khi thiếu jurisdiction'
assert r['confirmation_prompt'] is not None
print('✅ DoD 3 PASS')


[Q12 | Thiếu jurisdiction | DoD-3]
CÂU HỎI: Điều kiện chuyển mục đích sử dụng đất nông nghiệp sang đất ở là gì?
⚠️  CẦN XÁC NHẬN:
Bất động sản / đất đai của bạn thuộc tỉnh/thành phố nào? (TP. Hồ Chí Minh / Đồng Nai / địa phương khác)

⏱️  1.01s
✅ DoD 3 PASS


## 6. DoD 4 — Negative test: khai sinh TP.HCM vs Đồng Nai

In [16]:
r = ask(
    'Quy định đăng ký khai sinh tại TP.HCM khác Đồng Nai như thế nào?',
    label='Q13 | Negative — khai sinh địa phương | DoD-4'
)
results.append(r)
# Khai sinh là thủ tục toàn quốc → pipeline trả về empty context (chưa có data Hộ tịch)
# hoặc nói rõ không có sự khác biệt. Không được bịa sự khác biệt.
print('\n⚠️  Kiểm tra thủ công DoD 4: câu trả lời không được bịa sự khác biệt địa phương không tồn tại')


[Q13 | Negative — khai sinh địa phương | DoD-4]
CÂU HỎI: Quy định đăng ký khai sinh tại TP.HCM khác Đồng Nai như thế nào?


📊 LCCIDs: 0  |  Top-k: 0  |  Context: ~0 tokens

💬 TRẢ LỜI:
Không tìm thấy văn bản pháp luật liên quan đến câu hỏi này.

📌 CITATIONS parsed: 0 (LLM có thể dùng format tắt — kiểm tra thủ công)

⏱️  2.13s

⚠️  Kiểm tra thủ công DoD 4: câu trả lời không được bịa sự khác biệt địa phương không tồn tại


## 7. Gap 3 — câu hỏi cần traversal [:IMPLEMENTS]

In [17]:
r = ask(
    'Nghị định 102/2024/NĐ-CP hướng dẫn thi hành Luật Đất đai 2024 quy định gì về chuyển mục đích sử dụng đất tại TP.HCM?',
    label='Q14 | Gap3 — NĐ 102 CMĐSDĐ TP.HCM'
)
results.append(r)


[Q14 | Gap3 — NĐ 102 CMĐSDĐ TP.HCM]
CÂU HỎI: Nghị định 102/2024/NĐ-CP hướng dẫn thi hành Luật Đất đai 2024 quy định gì về chuyển mục đích sử dụng đất tại TP.HCM?


📊 LCCIDs: 13  |  Top-k: 22  |  Context: ~5851 tokens

💬 TRẢ LỜI:
## Quy định về chuyển mục đích sử dụng đất tại TP.HCM theo Nghị định 102/2024/NĐ-CP

Dựa trên CONTEXT được cung cấp, thông tin trực tiếp về nội dung Nghị định 102/2024/NĐ-CP liên quan đến chuyển mục đích sử dụng đất tại TP.HCM **không có đầy đủ trong tài liệu được lập chỉ mục**. Tuy nhiên, có thể tổng hợp các quy định liên quan từ các văn bản hiện có như sau:

---

### 1. Căn cứ cho phép chuyển mục đích sử dụng đất

Căn cứ để cho phép chuyển mục đích sử dụng đất nông nghiệp sang đất ở đối với hộ gia đình, cá nhân là **quy hoạch sử dụng đất cấp huyện hoặc quy hoạch chung hoặc quy hoạch phân khu** theo pháp luật về quy hoạch đô thị và nông thôn đã được cơ quan có thẩm quyền phê duyệt.
[Điều 116, Khoản 5, Văn bản luat-dat-dai-2024]

---

### 2. Thẩm quyền cho phép chuyển mục đích sử dụng đất tại TP.HCM

**UBND cấp huyện** có thẩm quyền cho phép chuyển mục đích sử dụng đất đối với **cá nhân**. Riêng trường hợp chuyển mục đích

In [18]:
r = ask(
    'Bảng giá đất TP.HCM năm 2025 ảnh hưởng như thế nào đến tiền sử dụng đất khi chuyển mục đích?',
    label='Q15 | Gap2+Gap3 — Bảng giá đất TP.HCM'
)
results.append(r)


[Q15 | Gap2+Gap3 — Bảng giá đất TP.HCM]
CÂU HỎI: Bảng giá đất TP.HCM năm 2025 ảnh hưởng như thế nào đến tiền sử dụng đất khi chuyển mục đích?


📊 LCCIDs: 7  |  Top-k: 19  |  Context: ~4036 tokens

💬 TRẢ LỜI:
Dựa trên thông tin trong CONTEXT, tôi có thể cung cấp một số thông tin liên quan, nhưng **CONTEXT không chứa bảng giá đất cụ thể của TP.HCM năm 2025**, do đó không thể trả lời đầy đủ về tác động cụ thể của bảng giá đất TP.HCM năm 2025.

## Những gì CONTEXT cung cấp được:

### 1. Vai trò của bảng giá đất trong tính tiền sử dụng đất
Theo Luật Đất đai 2024, bảng giá đất được sử dụng để **tính tiền sử dụng đất** khi Nhà nước công nhận quyền sử dụng đất ở của hộ gia đình, cá nhân và khi **chuyển mục đích sử dụng đất** của hộ gia đình, cá nhân.
[Điều 159, Khoản 1, Điểm a, Văn bản luat-dat-dai-2024]

### 2. Nghĩa vụ tài chính khi chuyển mục đích
Khi người sử dụng đất có nhu cầu chuyển từ đất nông nghiệp sang đất phi nông nghiệp (phù hợp quy hoạch), **phải nộp tiền sử dụng đất** theo quy định của pháp luật.
[Điều 25, Khoản 3, Văn bản nghi-dinh-101-2024-nd-cp]

---

## Giới hạn của câu trả lời:

CONTEXT **không có** thông tin về:
-

## 8. Tổng kết kết quả

In [19]:
print('\n' + '='*70)
print('TỔNG KẾT PIPELINE E2E TEST — TASK-14')
print('='*70)

total = len(results)
confirmed = sum(1 for r in results if r['confirmation_needed'])
answered = total - confirmed
with_parsed_citations = sum(1 for r in results if not r['confirmation_needed'] and r['citations'])
avg_elapsed = sum(r['elapsed_seconds'] for r in results) / total if total else 0

print(f'  Tổng câu hỏi:             {total}')
print(f'  Câu trả lời được:         {answered}')
print(f'  Cần xác nhận jurisdiction: {confirmed}')
print(f'  Có citation (format chuẩn): {with_parsed_citations}/{answered}')
print(f'  Thời gian TB:             {avg_elapsed:.1f}s')
if total:
    print(f'  Max elapsed:              {max(r["elapsed_seconds"] for r in results):.1f}s')

print('\nChi tiết:')
for i, r in enumerate(results, 1):
    if r['confirmation_needed']:
        status = '⚠️  CONFIRM'
    elif r['citations']:
        status = f'✅ {len(r["citations"])} cite'
    else:
        status = '⚡ 0 cite*'
    print(f'  Q{i:02d}: {status:12} {r["elapsed_seconds"]:5.1f}s | LCCIDs={r["lccids_count"]:4d} | {r["question"][:55]}')

print('\n* 0 cite = LLM có thể dùng format tắt, kiểm tra thủ công')


TỔNG KẾT PIPELINE E2E TEST — TASK-14
  Tổng câu hỏi:             15
  Câu trả lời được:         14
  Cần xác nhận jurisdiction: 1
  Có citation (format chuẩn): 13/14
  Thời gian TB:             22.4s
  Max elapsed:              38.8s

Chi tiết:
  Q01: ✅ 12 cite     38.5s | LCCIDs=  13 | Điều kiện để chuyển mục đích sử dụng đất tại TP.HCM là 
  Q02: ✅ 5 cite      23.0s | LCCIDs=  13 | Hộ gia đình có được chuyển đất nông nghiệp sang đất ở t
  Q03: ✅ 4 cite      21.2s | LCCIDs=  13 | Hồ sơ xin chuyển mục đích sử dụng đất tại TP.HCM gồm nh
  Q04: ✅ 5 cite      21.0s | LCCIDs=  13 | Nghĩa vụ tài chính khi chuyển mục đích sử dụng đất sang
  Q05: ✅ 9 cite      38.8s | LCCIDs=  13 | Quy trình chuyển mục đích sử dụng đất nông nghiệp sang 
  Q06: ✅ 2 cite      13.6s | LCCIDs=  13 | Thời hạn giải quyết hồ sơ chuyển mục đích sử dụng đất t
  Q07: ✅ 10 cite     30.3s | LCCIDs=  13 | Điều kiện để được cấp Giấy chứng nhận quyền sử dụng đất
  Q08: ✅ 6 cite      30.3s | LCCIDs=  13 | Hồ sơ đăng ký cấp 

In [20]:
# Đóng clients
neo4j_driver.close()
print('Clients closed.')

Clients closed.
